# Big Sur Marathon

The course on USGS contour lines, with Heartbreak Hill drawn to the same scale. Run `fetch_week1.py day2` first.

## 1 · Settings

In [ ]:
DAY      = 2
PROMPT   = "Lines"
SLUG     = "lines-big-sur"
NOTEBOOK = "day02_lines_big_sur.ipynb"
TITLE    = "Big Sur\nMarathon"
CREDITS  = "Course facts: Big Sur International Marathon. Route: © OpenStreetMap contributors. Elevation and contours: USGS 3DEP."

## 2 · Load

In [ ]:
import matplotlib, matplotlib.pyplot as plt, numpy as np, pandas as pd
from scipy import ndimage
from mapkit import poster, publish
from mapkit.paths import PROCESSED

route = pd.read_csv(PROCESSED / "bigsur_profile.csv")
boston = pd.read_csv(PROCESSED / "boston_comm_ave.csv")
dem_file = np.load(PROCESSED / "bigsur_dem.npz")
dem = dem_file["dem_dm"] / 10.0
x0, y0, x1, y1 = dem_file["bbox_3857"]

## 3 · Analysis, every number on the map comes from here

In [ ]:
def flatten_bridges(df):
    """Elevation models read the canyon under a bridge, so we draw a straight deck across each one."""
    z = df.elev_ft.copy()
    z[df.bridge] = np.nan
    return z.interpolate(limit_direction="both")

route["z"] = flatten_bridges(route)
boston["z"] = boston.elev_ft.rolling(5, center=True, min_periods=1).mean()   # smooth out street level noise

def biggest_climb(df):
    """The largest rise from any low point to a later high point."""
    low = df.z.cummin()
    top = (df.z - low).idxmax()
    bottom = df.z[:top + 1].idxmin()
    return bottom, top, float(df.z[top] - df.z[bottom])

# Hurricane Point: from the Little Sur River to the top of the course
a = route[(route.dist_mi >= 9.5) & (route.dist_mi <= 12.5)]
hp_low, hp_top, hp_rise = biggest_climb(a)
hp_miles = route.dist_mi[hp_top] - route.dist_mi[hp_low]
# Heartbreak Hill: the last big climb on Commonwealth Avenue
b = boston[(boston.dist_mi >= 1.0) & (boston.dist_mi <= 2.0)]    # past the earlier Newton hills
h_low, h_top, h_rise = biggest_climb(b)
h_miles = boston.dist_mi[h_top] - boston.dist_mi[h_low]
ratio = hp_rise / h_rise
print(f"Route length       {route.dist_mi.max():.2f} mi  (official 26.2)")
print(f"Top of the course  {route.z.max():.0f} ft  (official 560)")
print(f"Hurricane Point    {hp_rise:.0f} ft over {hp_miles:.2f} mi, mile {route.dist_mi[hp_low]:.1f} to {route.dist_mi[hp_top]:.1f}  (official chart: 40 to 560 ft = 520)")
print(f"Heartbreak Hill    {h_rise:.0f} ft over {h_miles:.2f} mi  (published 88 to 91)")
print(f"Ratio              {ratio:.1f}x")
print(f"Total climbing     {route.z.diff().clip(lower=0).sum():.0f} ft  (published 2,182)")

## 4 · Map

In [ ]:
PAPER, OCEAN, CONTOUR, ROUTE = "#f5f0e4", "#cfe0e6", "#a98a66", "#d11a2a"
fig, ax = poster.page(PAPER)
fx0, fy0, fx1, fy1 = poster.fit_frame(ax, -122.15, 36.105, -121.72, 36.565)

# clean up bad pixels, and keep only the mainland
bad = dem > 1800
if bad.any():
    dem[bad] = ndimage.median_filter(dem, size=9)[bad]
labels, _ = ndimage.label(ndimage.binary_opening(dem > 1, iterations=2))
sizes = np.bincount(labels.ravel())
sizes[0] = 0
is_land = labels == sizes.argmax()

ax.add_patch(plt.Rectangle((fx0, fy0), fx1 - fx0, fy1 - fy0, color=OCEAN, zorder=0))
ax.imshow(np.dstack([np.full(dem.shape + (3,), matplotlib.colors.to_rgb(PAPER)), is_land.astype(float)]),
          extent=(x0, x1, y0, y1), origin="upper", zorder=1)

# contour lines: thin every 100 m, heavier every 500 m
smooth = ndimage.gaussian_filter(np.where(is_land, dem, 0), 1.6)
gx = np.linspace(x0, x1, dem.shape[1])
gy = np.linspace(y1, y0, dem.shape[0])
top = int(smooth.max() // 100 + 1) * 100
thin = [v for v in range(100, top, 100) if v % 500]
thick = list(range(500, top, 500))
ax.contour(gx, gy, smooth, levels=thin, colors=CONTOUR, linewidths=0.32, alpha=0.8, zorder=2)
ax.contour(gx, gy, smooth, levels=thick, colors=CONTOUR, linewidths=0.8, zorder=2)
ax.contour(gx, gy, is_land.astype(float), levels=[0.5], colors="#6f8b96", linewidths=0.8, zorder=3)
ax.set_xlim(fx0, fx1); ax.set_ylim(fy0, fy1)

# the course, with the big climb standing out
rx, ry = poster.merc(route.lon, route.lat)
ax.plot(rx, ry, color="white", lw=5.4, solid_capstyle="round", zorder=5)
ax.plot(rx, ry, color=ROUTE, lw=2.7, solid_capstyle="round", zorder=6)
climb = route.loc[hp_low:hp_top]
cx, cy = poster.merc(climb.lon, climb.lat)
ax.plot(cx, cy, color="#7a0014", lw=5.2, solid_capstyle="round", zorder=7)

def mark(mile_at, text, side="right", bold=False):
    i = (route.dist_mi - mile_at).abs().idxmin()
    mx, my = poster.merc(route.lon[i], route.lat[i])
    ax.scatter([mx], [my], s=44, color="white", edgecolor=ROUTE, lw=1.4, zorder=9)
    gap = (fx1 - fx0) * 0.016
    poster.halo_text(ax, mx + (gap if side == "right" else -gap), my, text, color="#2a2420", size=13.5,
                     path=poster.BARLOW_BOLD if bold else poster.BARLOW, halo=PAPER, va="center", zorder=9,
                     ha="left" if side == "right" else "right")

mark(0, "Start: Big Sur", bold=True)
mark(11, "Hurricane Point climb, miles 10 to 12", bold=True)
mark(13.2, "Bixby Bridge, a grand piano plays here")
mark(route.dist_mi.max(), "Finish: Carmel", side="left", bold=True)

# the elevation profile along the bottom, with Heartbreak Hill drawn to the same scale
strip = fig.add_axes([0.06, 0.082, 0.88, 0.1], facecolor="none")
strip.fill_between(route.dist_mi, route.z, color=ROUTE, alpha=0.2, lw=0)
strip.plot(route.dist_mi, route.z, color=ROUTE, lw=1.3)
strip.fill_between(climb.dist_mi, climb.z, color="#7a0014", lw=0)
hb = boston.loc[h_low:h_top]
hb_start = 28.0
hb_x = hb_start + hb.dist_mi - hb.dist_mi.iloc[0]
strip.fill_between(hb_x, hb.z - boston.z[h_low], color="#7a0014", lw=0)
strip.set_xlim(0, 33.5); strip.set_ylim(0, 640)
strip.axhline(0, color="#2a2420", lw=0.7)
for s in strip.spines.values():
    s.set_visible(False)
strip.set_xticks([0, 5, 10, 15, 20, 26.2], ["0", "5", "10", "15", "20", "26.2 miles"], fontproperties=poster.font(poster.BARLOW, 11), color="#5c5247")
strip.tick_params(axis="x", length=0, pad=3)
strip.set_yticks([])
strip.patch.set_alpha(0)
strip.text(route.dist_mi[hp_top] + 0.35, 590, f"Hurricane Point: {hp_rise:.0f} ft", color="#7a0014", fontproperties=poster.font(poster.BARLOW_BOLD, 13), va="center")
strip.text(hb_start - 0.1, 165, f"Boston's Heartbreak Hill\n{h_rise:.0f} ft, same scale", color="#7a0014", fontproperties=poster.font(poster.BARLOW_BOLD, 12), va="bottom", linespacing=1.25)

poster.title(fig, TITLE, size=62, x=0.06, y=0.955, color="#231e1a", path=poster.SLAB, sub_path=poster.BARLOW,
             subtitle="The world's largest rural marathon: 26.2 miles\non California's Highway 1, with one climb over\nfive times the size of Heartbreak Hill.", sub_size=16, sub_color="#3d3730")
poster.credit(fig, CREDITS, color="#5e564b", path=poster.BARLOW, name_path=poster.BARLOW_BOLD, name_color="#231e1a")

## 5 · Export

In [ ]:
publish.save(fig, day=DAY, slug=SLUG, prompt=PROMPT, title=TITLE.replace("\n", " "),
             subtitle=f"The course on contour lines, and a climb over five times Heartbreak Hill", series="city", tool="Python",
             notebook=NOTEBOOK, credits=CREDITS)